In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [2]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=False)

In [3]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from controllers.directional_trading.elitesmugplug import SmugPlug3LiteConfig  # Fixed import
import datetime
from decimal import Decimal


# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "WLD-USDT"
interval = "1m"
total_amount_quote = 1000
max_executors_per_side = 2
take_profit = 0.29
stop_loss = 0.06
trailing_stop_activation_price = 0.05
trailing_stop_trailing_delta = 0.019
time_limit = 60 * 60 * 8  # 8 hours
cooldown_time = 60  # 1 minute

# 3lite specific parameters
ema_fast = 8  # Range: 3-8
ema_medium = 21  # Range: 8-21
ema_slow = 34  # Range: 13-34
atr_length = 10  # Range: 10-21
atr_multiplier = 0.9  # Range: 0.8-1.5
volume_surge = 4.5  # Range: 1.5-3.0
rsi_period = 13  # Range: 8-21


# Creating the instance of the configuration and the controller
config = SmugPlug3LiteConfig(
    id=f"3litesmugplug_{connector_name}_{interval}_{trading_pair}",  # Added id parameter
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    ema_fast=ema_fast,
    ema_medium=ema_medium,
    ema_slow=ema_slow,  # Note: changed from ema_long to ema_slow to match config
    atr_length=atr_length,
    atr_multiplier=atr_multiplier,
    volume_surge=Decimal(volume_surge),
    rsi_period=rsi_period,
    total_amount_quote=Decimal(total_amount_quote),
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(
        activation_price=Decimal(trailing_stop_activation_price),
        trailing_delta=Decimal(trailing_stop_trailing_delta)
    ),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
)

In [4]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 1, 1).timestamp())
end = int(datetime.datetime(2025, 1, 30).timestamp())


backtesting_result = await backtesting.run_backtesting(config, start, end, "1m")

2025-02-04 15:36:59,073 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x14f5e7100>
2025-02-04 15:36:59,075 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x14f4c6c20>, 403648.469018916)])']
connector: <aiohttp.connector.TCPConnector object at 0x14f5e7370>
2025-02-04 15:37:14,787 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x14f82b940>
2025-02-04 15:37:14,791 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x14f80aec0>, 403664.179217916)])']
connector: <aiohttp.connector.TCPConnector object at 0x14f82b970>
2025-02-04 15:37:28,066 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x14f82bb80>
2025-02-04 15:37:28,069 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseH

In [5]:
# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
backtesting_result.get_backtesting_figure()


Net PNL: $61.38 (6.14%) | Max Drawdown: $-50.06 (-5.02%)
Total Volume ($): 51000.00 | Sharpe Ratio: -0.15 | Profit Factor: 1.41
Total Executors: 51 | Accuracy Long: 0.56 | Accuracy Short: 0.43
Close Types: Take Profit: 0 | Stop Loss: 1 | Time Limit: 49 |
             Trailing Stop: 1 | Early Stop: 0



In [6]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df.head()

,id,timestamp,type,close_timestamp,close_type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,controller_id,side
0,AegtaT525RciwvW2SdUc89Ga4jJG4nNTqidVeZq1Be1y,1735855800,position_executor,1735863000,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'AegtaT525RciwvW2SdUc89Ga4jJG4nNTqidVeZ...,-0.0064456399437414317599159829796917620114982...,-3.2228199718707157828134768351446837186813354...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 2.2885, 'level_id': None, 'sid...",None,SELL
1,HrQAhnnSrFVdK75do1tyKy6FMgH9ATG6EFWRhmJryRN7,1735922820,position_executor,1735930020,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'HrQAhnnSrFVdK75do1tyKy6FMgH9ATG6EFWRhm...,-0.0073145075328378654500038891228541615419089...,-3.65725376641893262785742990672588348388671875,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 2.3989, 'level_id': None, 'sid...",None,SELL
2,FNUP1NCW4pj13eaf3NhoskmRPVTVUpL8wb6xX16JEHVH,1735931820,position_executor,1735939020,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'FNUP1NCW4pj13eaf3NhoskmRPVTVUpL8wb6xX1...,0.00230818446198626306409273922781721921637654...,1.15409223099313162919088426860980689525604248...,0.29999999999999998889776975374843459576368331...,1000.0000000000001136868377216160297393798828125,False,False,"{'close_price': 2.414, 'level_id': None, 'side...",None,BUY
3,F4LWbJPkShgBEx1sjGYMtrUmfEdBkDsnBMUZgr6gu75k,1735945200,position_executor,1735952400,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': 'F4LWbJPkShgBEx1sjGYMtrUmfEdBkDsnBMUZgr...,-0.0094381742738585787388938541653260472230613...,-4.7190871369292901604808321280870586633682250...,0.29999999999999998889776975374843459576368331...,1000.0000000000001136868377216160297393798828125,False,False,"{'close_price': 2.3887, 'level_id': None, 'sid...",None,BUY
4,3xQuKQSVU4spWTftc2puS1Dqw1msdnC4kQYb69kE9JzZ,1735948740,position_executor,1735955940,CloseType.TIME_LIMIT,RunnableStatus.TERMINATED,{'id': '3xQuKQSVU4spWTftc2puS1Dqw1msdnC4kQYb69...,-0.0131632307819877329702062951355401310138404...,-6.5816153909938668320478427631314843893051147...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 2.3815, 'level_id': None, 'sid...",None,BUY


### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [7]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [8]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT